In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from src.tasks.data_loader import load_mante_data
from src.models.ctrnn import CTRNN

In [ ]:
# ----------------------------
# Configuration
# ----------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

HIDDEN_SIZES = [20, 50, 100]

PDM_DIR = Path("E:\TUM\3sem\NeuroAI\project\data_generation\new\perceptual")
CDM_DIR = Path("E:\TUM\3sem\NeuroAI\project\data_generation\new\context")

RESULTS_DIR = Path("results")
WEIGHTS_DIR = RESULTS_DIR / "weights"

OUTPUT_DIM = 3

In [ ]:
class WrappedCTRNN(nn.Module):

    def __init__(self, input_dim, hidden_size):
        super().__init__()

        self.model = CTRNN(
            input_size=input_dim,
            hidden_size=hidden_size,
            output_size=OUTPUT_DIM,
        )

    def forward(self, x):
        outputs, _, hidden = self.model(
            x,
            return_dynamics=True,
        )
        return outputs, hidden

In [ ]:
def masked_cross_entropy(outputs, targets, periods):

    mask = periods == 2

    if mask.sum() == 0:
        return torch.tensor(
            0.0,
            device=outputs.device,
        )

    return F.cross_entropy(
        outputs[mask],
        targets[mask],
    )


def masked_accuracy(outputs, targets, periods):

    mask = periods == 2

    preds = outputs[mask].argmax(1)

    return (preds == targets[mask]).float().mean().item()

In [ ]:
def evaluate_model(model, loader):

    model.eval()

    losses = []
    accuracies = []

    with torch.no_grad():

        for obs, labels, periods, cohs, ctxs in loader:

            obs = obs.to(DEVICE)
            labels = labels.to(DEVICE)
            periods = periods.to(DEVICE)

            outputs, _ = model(obs)

            loss = masked_cross_entropy(
                outputs,
                labels,
                periods,
            )

            acc = masked_accuracy(
                outputs,
                labels,
                periods,
            )

            losses.append(loss.item())
            accuracies.append(acc)

    return np.mean(losses), np.mean(accuracies)

In [ ]:
loaders = {}

tasks = {
    "pdm": {
        "dir": PDM_DIR,
        "input_dim": 3,
    },
    "cdm": {
        "dir": CDM_DIR,
        "input_dim": 7,
    },
}

for task, info in tasks.items():

    test_path = info["dir"] / "test_uniform.npz"

    if not test_path.exists():
        test_path = info["dir"] / "test_01.npz"

    loaders[task] = load_mante_data(
        str(test_path),
        batch_size=2048,
        shuffle=False,
        input_dim=info["input_dim"],
    )

In [ ]:
results = {}

for h in HIDDEN_SIZES:

    print("=" * 60)
    print(f"Hidden size = {h}")

    for task in ["pdm", "cdm"]:

        input_dim = tasks[task]["input_dim"]

        model = WrappedCTRNN(
            input_dim=input_dim,
            hidden_size=h,
        )

        weight_path = WEIGHTS_DIR / f"ctrnn_{task}_h{h}.pth"

        model.load_state_dict(
            torch.load(
                weight_path,
                map_location=DEVICE,
            )
        )

        model.to(DEVICE)

        test_loss, test_acc = evaluate_model(
            model,
            loaders[task],
        )

        print(
            f"{task.upper()} "
            f"Loss={test_loss:.4f} "
            f"Acc={test_acc:.4f}"
        )

        results[f"{task}_{h}_test_loss"] = test_loss
        results[f"{task}_{h}_test_acc"] = test_acc

In [ ]:
for h in HIDDEN_SIZES:

    result_file = RESULTS_DIR / f"results_{h}.npz"

    data = dict(np.load(result_file, allow_pickle=True))

    task = "pdm"
    data[f"{task}_{h}_test_loss"] = results[f"{task}_{h}_test_loss"]
    data[f"{task}_{h}_test_acc"] = results[f"{task}_{h}_test_acc"]

    task = "cdm"
    data[f"{task}_{h}_test_loss"] = results[f"{task}_{h}_test_loss"]
    data[f"{task}_{h}_test_acc"] = results[f"{task}_{h}_test_acc"]

    np.savez_compressed(result_file, **data)

    print(f"Updated {result_file}")